# GymMate AI - Qwen 2.5 1.5B Fine-Tuning
Notebook ini dikustomisasi khusus untuk melatih parser gym log.

In [ ]:
%%capture
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir "unsloth[colab-new]" @ git+https://github.com/unslothai/unsloth.git
!pip install -U trl peft accelerate bitsandbytes xformers

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Sesuai dengan VRAM laptop target
dtype = None # Auto deteksi float16 untuk T4
load_in_4bit = True # Wajib 4bit biar muat di T4

# Kita pakai Qwen 2.5 1.5B versi Instruct
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Pasang LoRA (Low-Rank Adaptation)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank optimal untuk task ini
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset

# Terapkan format Qwen 2.5
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

# UPLOAD FILE 'dataset_train.jsonl' SEBELUM RUN INI!
dataset = load_dataset("json", data_files="dataset_train.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True,)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # ~2-3 epochs untuk dataset kita
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# MENCEGAH AI MENGHAFAL PROMPT (Penting!)
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

trainer_stats = trainer.train()

In [ ]:
# EXPORT KE GGUF UNTUK LLAMA.CPP LOKAL (MX350)
print("Mengekspor model ke GGUF Q4_K_M...")
model.save_pretrained_gguf("gymmate-qwen-1.5b", tokenizer, quantization_method = "q4_k_m")
print("Selesai! Silakan download file gymmate-qwen-1.5b-unsloth-Q4_K_M.gguf dari folder file di samping.")